In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

DATA_PATH = "cxr_df.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
FINDINGS = {
    "Pneumonia": [
        "pneumonia"
    ],

    "Pleural Effusion": [
        "pleural effusion",
        "pleural fluid",
        "effusions"
    ],

    "Pneumothorax": [
        "pneumothorax",
        "pneumothoraces"
    ],

    "Atelectasis": [
        "atelectasis"
    ],

    "Consolidation": [
        "consolidation"
    ],

    "Pulmonary Edema": [
        "pulmonary edema",
        "pulmonary oedema"
    ],

    "Cardiomegaly": [
        "cardiomegaly",
        "enlarged cardiac silhouette",
        "enlarged heart"
    ],

    "Opacity": [
        "opacity",
        "opacities",
        "opacification"
    ],

    "Nodule": [
        "nodule",
        "nodular"
    ],

    "Mass": [
        "mass",
        "mass-like"
    ],

    "Fracture": [
        "fracture",
        "fractures"
    ],

    "Emphysema": [
        "emphysema"
    ],

    "Fibrosis": [
        "fibrosis",
        "fibrotic"
    ],

    "Pleural Thickening": [
        "pleural thickening"
    ]
}

In [ ]:
text = df["text"].fillna("").str.lower()

counts = {}

for finding, terms in FINDINGS.items():
    pattern = "|".join(re.escape(term) for term in terms)

    counts[finding] = text.str.contains(
        pattern,
        regex=True,
        na=False
    ).sum()

finding_counts = (
    pd.Series(counts)
    .sort_values(ascending=False)
)

print(finding_counts)

In [ ]:
# Percentage of reports mentioning each finding
finding_percentage = (
    finding_counts / len(df) * 100
).round(2)

summary = pd.DataFrame({
    "Count": finding_counts,
    "Percentage": finding_percentage
})

print(summary)

In [ ]:
plt.figure(figsize=(10, 6))

finding_counts.sort_values().plot(kind="barh")

plt.xlabel("Number of reports")
plt.ylabel("Finding")
plt.title("Chest X-ray Finding Frequency")
plt.tight_layout()

plt.show()

In [ ]:
def classify_finding(text, terms):
    """
    Returns:
        1  -> positive
        0  -> explicitly negative
        -1 -> uncertain
        None -> not mentioned
    """

    if not isinstance(text, str):
        return None

    text = text.lower()

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sentence in sentences:

        # Check whether the finding is mentioned
        matched = any(
            re.search(
                r'\b' + re.escape(term) + r'\b',
                sentence
            )
            for term in terms
        )

        if not matched:
            continue

        # -------------------------
        # UNCERTAIN
        # -------------------------
        uncertainty_patterns = [
            r'\bpossible\b',
            r'\bpossibly\b',
            r'\bmay represent\b',
            r'\bmay be\b',
            r'\bmay develop\b',
            r'\bmay be developing\b',
            r'\bcannot exclude\b',
            r'\bcannot be excluded\b',
            r'\bquestion of\b',
            r'\bsuspicious for\b',
            r'\bsuggestive of\b',
            r'\bconsistent with\b.*\bpossible\b',
            r'\bcoexisting\b.*\bpossible\b',
        ]

        if any(
            re.search(pattern, sentence)
            for pattern in uncertainty_patterns
        ):
            return -1

        # -------------------------
        # NEGATIVE
        # -------------------------
        negative_patterns = [
            r'\bno evidence of\b',
            r'\bno\b',
            r'\bwithout\b',
            r'\bnegative for\b',
            r'\babsence of\b',
            r'\babsent\b',
            r'\bnot seen\b',
            r'\bnot identified\b',
            r'\bthere is no\b',
            r'\bthere are no\b',
            r'\bno clear evidence of\b',
        ]

        if any(
            re.search(pattern, sentence)
            for pattern in negative_patterns
        ):
            return 0

        # -------------------------
        # POSITIVE
        # -------------------------
        return 1

    return None

In [ ]:
NEGATION_PHRASES = [
    "no evidence of",
    "no",
    "without",
    "negative for",
    "absence of",
    "absent",
    "not seen",
    "not identified",
    "there is no",
    "there are no"
]

UNCERTAIN_PHRASES = [
    "possible",
    "possibly",
    "may represent",
    "cannot exclude",
    "cannot be excluded",
    "question of",
    "suspicious for",
    "suggestive of",
    "likely"
]

In [ ]:
test_reports = [
    "No evidence of pneumothorax.",
    "There is a small right pneumothorax.",
    "Possible pneumothorax cannot be excluded.",
    "No pleural effusion.",
    "Small bilateral pleural effusions are present.",
    "Coexisting pneumonia is possible given clinical history.",
    "New basal consolidation suggesting pneumonia may be developing."
]

for report in test_reports:

    if "pneumonia" in report.lower():
        finding = "Pneumonia"
    elif "pleural" in report.lower():
        finding = "Pleural Effusion"
    else:
        finding = "Pneumothorax"

    result = classify_finding(
        report,
        FINDINGS[finding]
    )

    print(result, "->", report)

In [ ]:
for finding in ["Pneumonia", "Pleural Effusion", "Pneumothorax"]:

    print("\n" + "=" * 70)
    print(finding)
    print("=" * 70)

    terms = FINDINGS[finding]

    count = 0

    for _, row in df.iterrows():

        result = classify_finding(row["text"], terms)

        if result is not None:

            print(f"\nLabel: {result}")
            print(row["text"][:500])

            count += 1

        if count >= 5:
            break

In [ ]:
for finding, terms in FINDINGS.items():
    df[f"label_{finding}"] = df["text"].fillna("").apply(
        lambda x: classify_finding(x, terms)
    )

In [ ]:
label_columns = [
    col for col in df.columns
    if col.startswith("label_")
]

label_summary = {}

for col in label_columns:
    counts = df[col].value_counts(dropna=False)

    label_summary[col.replace("label_", "")] = {
        "Positive": counts.get(1.0, 0),
        "Negative": counts.get(0.0, 0),
        "Uncertain": counts.get(-1.0, 0),
        "Not Mentioned": counts.get(np.nan, 0)
    }

label_summary_df = pd.DataFrame(label_summary).T

label_summary_df

In [ ]:
label_summary_df["Usable"] = (
    label_summary_df["Positive"] +
    label_summary_df["Negative"]
)

label_summary_df["Positive % of Mentioned"] = (
    label_summary_df["Positive"] /
    (
        label_summary_df["Positive"] +
        label_summary_df["Negative"] +
        label_summary_df["Uncertain"]
    ) * 100
).round(2)

label_summary_df

In [ ]:
import os

path_exists = df["path"].apply(os.path.exists)

print("Total rows:", len(df))
print("Images found:", path_exists.sum())
print("Images missing:", (~path_exists).sum())
print("Image availability:", round(path_exists.mean() * 100, 2), "%")

In [ ]:
print(df.loc[~path_exists, "path"].head(10).tolist())